In [1]:
import numpy as np
import pandas as pd

from bids import BIDSLayout
layout = BIDSLayout('bids_dataset')

In [2]:
# select subjects with acceptable EEG impedances
participants = layout.get(suffix = 'participants', extension = 'tsv')[0].get_df()
high_imp = participants.notes.str.contains('exceeded') == True
sub_nums = participants.participant_id.str.extract(r'sub-(\w+)')
good_subs = sub_nums[~high_imp].iloc[:, 0].to_list()
print('Excluded', sub_nums[high_imp][0].to_list())
print('%d subjects remaining.'%len(good_subs))

info = participants[participants.participant_id.isin(['sub-%s'%s for s in good_subs])]
info.head()

Excluded ['08', '19']
20 subjects remaining.


,participant_id,age,sex,hand,weight,height,notes
0,sub-01,19,M,R,NaN,NaN,NaN
1,sub-02,19,F,L,NaN,NaN,NaN
2,sub-03,19,F,R,NaN,NaN,NaN
3,sub-04,21,F,R,NaN,NaN,NaN
4,sub-05,20,M,R,NaN,NaN,NaN


In [3]:
(info.sex == 'M').sum()

12

In [4]:
(info.sex == 'F').sum()

8

In [5]:
(info.hand == 'R').sum()

18

In [6]:
(info.hand == 'L').sum()

2

In [7]:
(info.hand == 'A').sum()

0

In [8]:
info.age.mean()

19.4

In [9]:
info.age.min(), info.age.max()

(18, 21)

In [10]:
trial_counts = dict(streaming = [], discrimination = [])
for sub in good_subs:
    for f in layout.get(subject = sub, suffix = 'events', extension = 'tsv'):
        df = f.get_df()
        df = df[df.tone_num == 1].reset_index()
        trial_counts[f.entities['task']].append(df.shape[0])
df = pd.DataFrame(trial_counts)
df

,streaming,discrimination
0,99,100
1,100,100
2,100,100
3,95,100
4,91,100
5,100,100
6,99,100
7,100,100
8,100,100
9,100,100


In [11]:
dropped_trials = (df.streaming.max() - df.streaming).sum()
dropped_trials # across all participants

25